# Session 1 — Data Exploration

**Goal:** Understand the Telco Customer Churn dataset before building any models.

Notebooks are a great way to explore and gain an understanding of your data.

Key questions to answer:
- What does the target variable look like? (Class balance)
- Which features might be strong predictors of churn?
- Are there any data quality issues we need to handle?

In [0]:
%run ../utils/config

## Understand the shape of the data
Look at the number of rows and columns as well as a sampling of the data in the table

In [0]:
df = spark.table(f"00_shared.telco_churn")

print(f"Shape: {df.count():,} rows × {len(df.columns)} columns")
display(df.limit(10))

## Target Variable: Churn

First, let's check the class balance. A significant imbalance will affect how we choose and evaluate our models.

In [0]:
from pyspark.sql import functions as F

churn_counts = df.groupBy("Churn").count().orderBy("Churn")
display(churn_counts)

# Calculate churn rate
total = df.count()
churned = df.filter(F.col("Churn") == "Yes").count()
print(f"\nChurn rate: {churned/total:.1%}  ({churned:,} / {total:,} customers)")
print("\nℹ️  ~26% churn rate — moderately imbalanced.")
print("   This means accuracy alone is misleading — we'll focus on F1 and Recall.")

## Contract Type vs Churn

Intuitively, customers on month-to-month contracts would be more likely to churn than those locked into annual contracts.

_Note:_ The following cells have visualization tabs in the output in addition to the Table data.  Notebook visualizations can help in your Exploratory Data Analysis.

In [0]:
contract_churn = (
    df.groupBy("Contract", "Churn")
      .count()
      .orderBy("Contract", "Churn")
)
display(contract_churn)

Databricks visualization. Run in Databricks to view.

## Numeric Feature Distributions

`MonthlyCharges`, `TotalCharges`, and `tenure` are numeric features.
Compare their distributions for churned vs non-churned customers.

In [0]:
numeric_stats = df.groupBy("Churn").agg(
    F.round(F.mean("tenure"), 1).alias("avg_tenure"),
    F.round(F.mean("MonthlyCharges"), 2).alias("avg_monthly_charges"),
    F.round(F.mean("TotalCharges"), 2).alias("avg_total_charges"),
    F.count("*").alias("count")
).orderBy("Churn")

# Bar chart: compare avg_tenure, avg_monthly_charges, avg_total_charges by Churn
display(numeric_stats)

print("\nKey insight: churned customers have much shorter tenure on average.")
print("Short tenure + high monthly charges = high churn risk signal.")

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

## Data Quality: Null Values

`TotalCharges` can be null for brand-new customers (tenure = 0).
Our feature pipeline will handle this with median imputation.

In [0]:
null_counts = df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df.columns
]).toPandas().T.rename(columns={0: "null_count"})
null_counts = null_counts[null_counts["null_count"] > 0]

if len(null_counts) == 0:
    print("✓ No null values found.")
else:
    print("Columns with nulls:\n")
    print(null_counts)
    print("\n→ TotalCharges nulls = new customers. Our pipeline imputes with the median.")

## Key Takeaways

Before we build any model, we know:

1. **Class imbalance**: ~26% churn. Use F1/Recall, not accuracy.
2. **Strong predictors**: Contract type, tenure, MonthlyCharges
3. **Data quality**: TotalCharges has some nulls → median imputation needed
4. **Business context**: Month-to-month customers churn at ~3× the rate of annual customers


## Bonus: Explore with Genie

You've just queried the churn dataset using PySpark. **Genie** lets you ask the same questions — and new ones — in plain English, with no SQL required.

This is the same `telco_churn` table your model will train on, now surfaced for natural language analytics. It demonstrates that a single governed data asset in Unity Catalog can serve both your ML pipelines and your business analysts.

**Open the workshop Genie Space using the link generated in the next cell, then try the questions below.**

In [ ]:
workspace_url = spark.conf.get("spark.databricks.workspaceUrl")
genie_url = f"https://{workspace_url}/genie/rooms/{genie_space_id}"
displayHTML(f'''
<div style="padding:12px; background:#f0f4ff; border-left:4px solid #4c6ef5; border-radius:4px; font-family:sans-serif;">
  <strong>Workshop Genie Space</strong><br/>
  <a href="{genie_url}" target="_blank" style="font-size:15px;">
    Open: Telco Churn — Workshop ↗
  </a>
</div>
''')

### Sample questions to try

These mirror the insights you just found in code — now ask them in natural language:

**Match what you just explored:**
- *"What percentage of customers churned?"*
- *"Which contract type has the highest churn rate?"*
- *"Compare average tenure for churned vs retained customers"*

**Go further — segment and target:**
- *"Show me month-to-month customers on fiber optic paying more than $70/month"*
- *"Which payment methods have the highest churn rates?"*
- *"How does having tech support affect churn rate?"*

**Discussion:** A business analyst can now explore the same data your model trains on — with no SQL or Python. How does this change who can act on churn risk in your organization?

➡️ Next: [02_messy_notebook]($./02_messy_notebook) (instructor demo) → `03_refactored_pipeline.ipynb`